# WeldFatigueChecker · 自采 40 张增量训练（Kaggle Notebook）
把 `raw_mine` 自采焊缝图（40 张）并入基于 120 轮权重的检测模型，并**采用持续学习模式，历史数据绝不重跑**。

## 前置准备（在 kaggle.com/code 网页操作）
1. 新建 Notebook → 右上角 **Settings → Accelerator 选 GPU（T4）** → 保存。
2. 上传文件到工作区 `/kaggle/working/`：
   - 本仓库的 `ml/` 整库（拖进 Notebook 左侧文件面板，会自动到 `/kaggle/working/ml`）；
   - 120 轮权重 `last.pt`（6.0M，从 AI Studio 旧工程 `work/runs/detect/weld_defect_5cls/weights/last.pt` 下载后上传到这里，放到 `/kaggle/working/last.pt`）。
   - ⚠️ 别用 AI Studio 那个 18M 的 `WeldDefectModel.pt`。
3. 上传本 `.ipynb` 后**从上到下依次运行**所有 cell。

In [ ]:
# Kaggle 已预装 torch+CUDA；补装 ultralytics 与 coremltools（导出 .mlpackage 需要）
!pip install -q ultralytics pyyaml coremltools
import torch, ultralytics
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)

In [ ]:
ML_DIR = '/kaggle/working/ml'        # ← weld_train.py 所在目录（ml/ 上传后的位置）
CKPT   = '/kaggle/working/last.pt'   # ← 120 权重绝对路径（上传到 /kaggle/working/ 下）
EPOCHS = 40   # 首次只训 40 张自采图：30~40 轮足够；若同时上传了源1~4全量数据，改 120
%cd {ML_DIR}

In [ ]:
import os
print('weld_train.py 存在 :', os.path.isfile('weld_train.py'))
print('raw_mine 图片数   :', len([f for f in os.listdir('raw_mine/images') if f.lower().endswith(('.jpg','.png','.jpeg'))]))
print('raw_mine 标签数   :', len([f for f in os.listdir('raw_mine/labels') if f.lower().endswith('.txt')]))
print('120 权重存在      :', os.path.isfile(CKPT))

In [ ]:
# 持续学习：首次（无 manifest）把 40 张当新增全量训一次建立基线；之后重跑只训新增，历史不重跑
!python weld_train.py --incremental --init-from {CKPT} --epochs {EPOCHS}

In [ ]:
import glob, os
runs = sorted(glob.glob('runs/detect/weld_defect_5cls*/weights/best.pt'))
print('最新训练产物      :', runs[-1] if runs else '未找到')
print('mlpackage 已导出  :', os.path.isdir('WeldDefectModel.mlpackage'))
print('已训图清单 manifest:', os.path.isfile('dataset/.trained_manifest.txt'))
if os.path.isfile('dataset/.trained_manifest.txt'):
    with open('dataset/.trained_manifest.txt') as f:
        print('  已训图数        :', len([l for l in f if l.strip()]))

In [ ]:
from IPython.display import FileLink
display(FileLink('WeldDefectModel.mlpackage'))
print('训练完成后右键/点击上面链接下载 WeldDefectModel.mlpackage 回 Xcode 工程替换即可。')
print('建议同时把 last.pt 下载到本机兜底（每个 epoch 已自动备份）')

## 以后继续补充自采图（持续学习，历史绝不重跑）
1. 把新图丢进 `raw_mine/images/` + 同名标签 `raw_mine/labels/`（YOLO 5 类格式）。
2. 重跑上面的**训练 Cell**，但去掉 `--init-from`，改为：
   ```python
   !python weld_train.py --incremental
   ```
   - 脚本自动借上次 `last.pt` 权重，只训【新增图】+ 抽 ≤200 张历史图回放防遗忘，默认 30 轮，数十分钟级。
3. 下载新 `WeldDefectModel.mlpackage` 回 Xcode 工程替换即可。

⚠️ 不要把 `dataset/` 整个删除（会丢 `.trained_manifest.txt`）；想彻底重排历史才删该 manifest 文件。